# 04 · Modelado: pronóstico, alerta de brote, riesgo y perfiles de comunas

**Proyecto:** Alerta Dengue Cali — Secretaría de Salud Pública Distrital de Santiago de Cali
**Autores:** Samuel Acevedo Mambuscay y Luis Carlos Tez — Universidad Libre, seccional Cali

Este cuaderno diseña, compara y evalúa los modelos del producto de datos. Todo el código
reutilizable está en `flows/05_ml_models.py`; aquí solo se llaman sus funciones, se
comparan alternativas y se justifican las decisiones que quedan congeladas en ese módulo.

| Pregunta del proyecto | Tipo | Modelo |
|---|---|---|
| ¿Cuántos casos habrá en las semanas t+1 a t+4? | Regresión (serie de tiempo) | Pronóstico semanal con intervalo |
| ¿La ciudad estará en brote? | Clasificación de cada semana | Pronóstico frente al cuartil 3 del canal |
| ¿Este caso tendrá signos de alarma o será grave? | Clasificación | Random Forest con datos de la primera consulta |
| ¿El paciente requerirá hospitalización? | Clasificación | El mismo diseño, otro objetivo |
| ¿Cómo se agrupan las 22 comunas? | Agrupamiento | K-means |

**Regla de validación:** las decisiones se toman con datos de entrenamiento o validación y
el conjunto de prueba se abre **una sola vez** al final de cada sección. La definición del
problema y las métricas están en `reports/problema_ml.md`.

In [ ]:
import importlib
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

from flows.common.config import settings
from flows.common.epidemiologia import GRUPOS_EDAD, grupo_edad
from flows.common.paths import FIGURES, SILVER, SILVER_CASOS, SILVER_SERIE

ml = importlib.import_module("flows.05_ml_models")

sns.set_theme(style="whitegrid", context="notebook")
FIGURES.mkdir(parents=True, exist_ok=True)


def guardar(fig, nombre):
    fig.savefig(FIGURES / f"{nombre}.png", dpi=150, bbox_inches="tight")

## 1. Pronóstico de casos semanales

**Datos:** serie semanal de Silver (314 semanas, 2020-2025). **Objetivo:** el cambio
relativo de los casos entre la semana actual *t* y la semana *t + h*, en escala
logarítmica; así un mismo modelo sirve para semanas de 30 y de 1.300 casos.
**Variables:** el nivel de la semana actual, los cambios de las 7 semanas anteriores
(8 semanas de historia), la posición en el año de la semana a pronosticar y la anomalía de
temperatura de la ciudad en las últimas 4 semanas (IDEAM; nula cuando faltan lecturas).

**Regla de elección:** en cada horizonte gana el menor MAE de validación, pero si otros
modelos quedan a menos del 1 % (empate) se prefiere el más simple: ingenuo, luego
tendencia, luego tendencia con clima y por último aprendizaje automático.

Las semanas interpoladas no se usan ni como origen ni como objetivo: la interpolación usa
datos posteriores y eso filtraría información del futuro.

In [ ]:
serie = pd.read_parquet(SILVER / SILVER_SERIE)
X, y, meta = ml.dataset_pronostico(serie, horizonte=1)
print(f"Muestras para h=1: {len(X)} | utilizables (sin semanas interpoladas): "
      f"{int(meta['usable'].sum())}")
X.join(y.rename("objetivo")).head()

**Partición temporal:** se entrena con los objetivos de 2020-2022, se eligen los modelos
con 2023 (validación) y se prueba con 2024-2025. Candidatos:

- `ingenuo`: la próxima semana tendrá los mismos casos que la actual (referencia).
- `tendencia_25/50/75`: extrapola la tendencia de las últimas 4 semanas, amortiguada.
- `tendencia_clima_25/50/75`: la misma tendencia más un efecto de la anomalía de
  temperatura de las últimas 4 semanas, ajustado con las semanas de entrenamiento que
  tienen lecturas del IDEAM. Sin temperatura, el pronóstico es el de la tendencia.
- `ridge`, `random_forest`, `gradient_boosting`: aprenden el cambio a partir de las
  variables de casos (no aceptan las semanas sin clima).
- `ensamble_rf_ingenuo`: promedio del Random Forest y el ingenuo (la propuesta inicial).

In [ ]:
comparacion = ml.comparar_pronostico(serie)
tabla_mae = comparacion.pivot(index="modelo", columns="horizonte", values="mae")
tabla_mejora = comparacion.pivot(index="modelo", columns="horizonte",
                                 values="mejora_vs_ingenuo_pct")
print(f"MAE (casos por semana) en la validación {settings.anio_validacion}:")
tabla_mae.sort_values(4).round(1)

In [ ]:
print("Mejora del MAE frente al ingenuo (%), validación:")
tabla_mejora.sort_values(4, ascending=False).round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for modelo, fila in tabla_mae.iterrows():
    destacado = modelo in ("ingenuo", "tendencia_25", "tendencia_50", "ensamble_rf_ingenuo")
    ax.plot(fila.index, fila.values, marker="o", lw=2.2 if destacado else 1,
            alpha=1 if destacado else 0.5, label=modelo)
ax.set(title=f"Error de pronóstico en la validación ({settings.anio_validacion})",
       xlabel="Horizonte (semanas)", ylabel="MAE (casos por semana)", xticks=[1, 2, 3, 4])
ax.legend(fontsize=8, ncol=2)
guardar(fig, "fig11_validacion_pronostico")

In [ ]:
elegidos = ml.elegir_por_horizonte(comparacion)
print("Elegidos por la validación:", elegidos)
print("Congelados en flows/05_ml_models.py:", ml.PRONOSTICO_MODELO_POR_HORIZONTE)
assert elegidos == ml.PRONOSTICO_MODELO_POR_HORIZONTE

**Decisión.** A una semana nada supera al ingenuo: la serie cambia muy poco de una semana
a la siguiente (autocorrelación 0,99). De 2 a 4 semanas gana la tendencia amortiguada
(25 % a 2-3 semanas y 50 % a 4 semanas), entre 5 % y 9 % mejor que el ingenuo en 2023.

Las versiones con temperatura quedan empatadas con la tendencia sin clima (diferencias de
0,01 a 0,06 casos): el coeficiente aprendido es prácticamente cero, porque la anomalía de
temperatura no aporta información sobre el crecimiento de los casos, como ya mostró el
análisis exploratorio. Por la regla de empate se queda el modelo sin clima.

Los modelos de aprendizaje automático (Random Forest, gradient boosting, Ridge) son
**peores** que el ingenuo, y el ensamble con el ingenuo solo reduce la pérdida. La razón:
en 2020-2022 solo hay una epidemia y en descenso, así que aprenden que "los casos tienden a
bajar" y fallan cuando la serie sube, como en el segundo semestre de 2023. Con más años
epidémicos en el entrenamiento esto podría cambiar; por ahora el modelo simple es el
honesto.

### Prueba final (2024-2025), una sola vez

Cada modelo elegido se entrena con 2020-2023 y pronostica todas las semanas de 2024-2025.
El intervalo del 80 % sale de los errores de validación, y la alerta se evalúa contra un
canal endémico construido solo con años anteriores a la prueba (2021 y 2022), para que el
canal tampoco vea el futuro.

In [ ]:
evaluacion, resumen = ml.evaluar_pronostico(serie)
tabla_prueba = pd.DataFrame(resumen["por_horizonte"]).T
print("Canal de referencia para la alerta:", resumen["canal_anios_base"])
tabla_prueba[["modelo", "n_semanas", "mae", "mae_ingenuo", "mejora_vs_ingenuo_pct", "rmse",
              "mape", "cobertura_intervalo_pct", "semanas_brote", "alerta_sensibilidad_pct",
              "alerta_precision_pct", "alerta_especificidad_pct"]].infer_objects().round(1)

In [ ]:
fig, ejes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for ax, h in zip(ejes, [1, 4], strict=True):
    datos = evaluacion[evaluacion["horizonte"] == h].sort_values("fecha_objetivo")
    ax.fill_between(datos["fecha_objetivo"], datos["limite_inferior"], datos["limite_superior"],
                    color="#90caf9", alpha=0.5, label="Intervalo 80 %")
    ax.plot(datos["fecha_objetivo"], datos["casos_objetivo"], color="black", lw=1.6,
            label="Casos observados")
    ax.plot(datos["fecha_objetivo"], datos["pronostico"], color="#1565c0", lw=1.4, ls="--",
            label=f"Pronóstico a {h} semana{'s' if h > 1 else ''}")
    ax.plot(datos["fecha_objetivo"], datos["q3"], color="#c62828", lw=1, ls=":",
            label="Q3 del canal (2021-2022)")
    m = resumen["por_horizonte"][h]
    ax.set(title=f"h = {h}: MAE {m['mae']:.0f} casos (ingenuo {m['mae_ingenuo']:.0f}); "
                 f"alerta: sensibilidad {m['alerta_sensibilidad_pct']:.0f} %, precisión "
                 f"{m['alerta_precision_pct']:.0f} %", ylabel="Casos por semana")
    ax.legend(fontsize=8, loc="upper right")
fig.suptitle("Prueba del pronóstico en 2024-2025 (semanas que el modelo no vio)", y=1.0)
guardar(fig, "fig12_prueba_pronostico")

En la prueba el modelo mejora al ingenuo en 1,7 %, 4,7 % y 6,3 % a 2, 3 y 4 semanas (MAE
de 46, 60 y 72 casos), con MAPE entre 14 % y 25 %. La alerta detecta entre 96 % y 98 % de
las semanas en brote con precisión de 96 % a 98 %; este resultado lo leemos con cautela: en
2024-2025 casi todas las semanas están muy lejos del umbral (todo 2024 en brote, casi todo
2025 debajo), así que la alerta solo se equivoca en las pocas semanas de transición.

**El intervalo del 80 % cubre solo entre 61 % y 67 % de las semanas:** 2023 fue un año
más tranquilo que 2024 y los errores de validación subestiman la incertidumbre de una
epidemia grande. Es una limitación que se informa en la ficha del modelo.

**Análisis complementario (no se usa para elegir).** ¿Cuánto habría aportado la
temperatura en la prueba? La comparación usa 2024-2025, así que solo sirve como evidencia
para revisar la decisión cuando haya más semanas con lecturas del IDEAM.

In [ ]:
con_clima = {1: "ingenuo", 2: "tendencia_clima_25", 3: "tendencia_clima_25",
             4: "tendencia_clima_50"}
_, resumen_clima = ml.evaluar_pronostico(serie, con_clima)
pd.DataFrame({
    "modelo elegido": {h: m["modelo"] for h, m in resumen["por_horizonte"].items()},
    "MAE elegido": {h: m["mae"] for h, m in resumen["por_horizonte"].items()},
    "MAE con temperatura": {h: m["mae"] for h, m in resumen_clima["por_horizonte"].items()},
    "MAE ingenuo": {h: m["mae_ingenuo"] for h, m in resumen["por_horizonte"].items()},
}).round(1)

Con temperatura, el error de 2024-2025 habría cambiado menos de 1 caso por semana (de 46,1
a 45,4, de 59,6 a 58,8 y de 72,0 a 71,5 a 2, 3 y 4 semanas): el aporte es despreciable y
confirma la decisión. El clima queda integrado en el pipeline como variable candidata,
para volver a evaluarlo cuando el IDEAM publique series más completas.

### Modelo definitivo

Se reentrena con todas las semanas observadas y pronostica desde la última semana de los
datos (semana 53 de 2025).

In [ ]:
paquete = ml.entrenar_pronostico_final(serie)
proximas = ml.pronosticar(paquete, serie)
proximas[["horizonte", "anio_epi", "semana_epi", "fecha_inicio", "modelo", "pronostico",
          "limite_inferior", "limite_superior", "q3", "zona_pronostico", "alerta"]].round(
    {"pronostico": 1, "limite_inferior": 1, "limite_superior": 1})

## 2. Riesgo de signos de alarma o dengue grave

**Datos:** los 13.966 casos del Distrito (2022-2023) con clasificación final; los
microdatos del INS no traen síntomas ni clasificación. **Objetivo:** `severo` = con signos
de alarma o dengue grave (39 %). **Variables:** solo lo que se sabe en la primera
consulta: edad, sexo, estrato, comuna, afiliación, migrante, desplazamiento reciente, días
desde el inicio de síntomas y los seis síntomas generales.

**Partición temporal:** el 20 % de casos más recientes es la prueba, como pasaría al usar
el modelo con casos nuevos.

In [ ]:
X, y, fechas = ml.load_dataset_riesgo()
X_tr, X_te, y_tr, y_te = ml.split_temporal(X, y, fechas)
print(f"Entrenamiento: {len(X_tr):,} casos hasta {fechas[X_tr.index].max().date()} "
      f"({y_tr.mean() * 100:.1f} % severos)")
print(f"Prueba: {len(X_te):,} casos desde {fechas[X_te.index].min().date()} "
      f"({y_te.mean() * 100:.1f} % severos)")
X_tr.head()

### Por qué se excluyen los signos de alarma

La clasificación "con signos de alarma" se define justamente por la presencia de esos
signos. Si entran como variables, el modelo solo "lee la definición" y el resultado es
engañoso:

In [ ]:
casos = pd.read_parquet(SILVER / SILVER_CASOS)
casos = casos[casos["severo"].notna()].reset_index(drop=True)
signos = [c for c in ml.RIESGO_EXCLUIDAS if c in casos and casos[c].dtype == "boolean"]
con_signos = casos[signos].astype("float64").loc[X_tr.index]
fuga = cross_val_score(make_pipeline(SimpleImputer(), LogisticRegression(max_iter=1000)),
                       con_signos, y_tr, cv=5, scoring="roc_auc")
ninguno = con_signos.fillna(0).sum(axis=1) == 0
print(f"AUC usando solo los {len(signos)} signos de alarma: {fuga.mean():.3f}")
print(f"Casos sin ningún signo de alarma: {int(ninguno.sum()):,}; de ellos severos: "
      f"{int(y_tr[ninguno].sum())}")

Un AUC de 1,0 no sirve para nada: en la primera consulta esos signos todavía no se conocen
o, si se conocen, el caso ya está clasificado. Por eso el modelo usa solo información
previa.

### Comparación de candidatos (validación cruzada de 5 pliegues, solo entrenamiento)

In [ ]:
comparacion_riesgo = ml.comparar_riesgo(X_tr, y_tr)
comparacion_riesgo.round(3)

Random Forest y gradient boosting quedan prácticamente empatados (0,620 y 0,623, dentro de
una desviación estándar) y ambos superan a la regresión logística (0,591) y a la
referencia (0,5). Se elige el Random Forest, que es el modelo propuesto en el proyecto y
es más fácil de explicar al equipo de vigilancia; la brecha entre entrenamiento y
validación (0,10) se controla con hojas de al menos 20 casos.

### Ajuste de hiperparámetros (búsqueda en malla)

In [ ]:
busqueda = ml.buscar_hiperparametros_riesgo(X_tr, y_tr)
resultados = (pd.DataFrame(busqueda.cv_results_)
              .sort_values("rank_test_score")
              [["params", "mean_test_score", "std_test_score"]].head(6))
print("Mejores parámetros:", busqueda.best_params_)
print("Congelados:", ml.RIESGO_PARAMS)
assert busqueda.best_params_ == ml.RIESGO_PARAMS
resultados.round(4)

Se congela la combinación ganadora: 400 árboles de profundidad máxima 10, hojas de al
menos 20 casos y 20 % de las variables por nodo. Las primeras combinaciones difieren en
menos de 0,001 de AUC (menos que la variación entre pliegues): el desempeño está limitado
por la información disponible en la primera consulta, no por los hiperparámetros.

### Umbral y niveles de riesgo

Con predicciones fuera de muestra del entrenamiento se fija el umbral de Youden (máxima
sensibilidad + especificidad) y los cortes de los niveles Bajo, Medio y Alto (tercios).

In [ ]:
pipeline, umbrales = ml.entrenar_riesgo(X_tr, y_tr)
umbrales

### Prueba final (una sola vez)

In [ ]:
prueba = ml.evaluar_riesgo(pipeline, X_te, y_te, umbrales)
print(json.dumps({k: v for k, v in prueba.items() if k != "por_subgrupo"}, indent=1,
                 ensure_ascii=False))

In [ ]:
proba = pipeline.predict_proba(X_te)[:, 1]
fpr, tpr, _ = roc_curve(y_te, proba)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(fpr, tpr, color="#c62828", lw=2, label=f"Random Forest (AUC {prueba['roc_auc']:.3f})")
ax1.plot([0, 1], [0, 1], color="grey", ls=":", label="Azar")
ax1.scatter(1 - prueba["especificidad"], prueba["sensibilidad"], color="black", zorder=3,
            label=f"Umbral {umbrales['umbral']:.2f}")
ax1.set(title="Curva ROC en la prueba", xlabel="1 - especificidad", ylabel="Sensibilidad")
ax1.legend(loc="lower right")
niveles = pd.Series(prueba["pct_positivos_por_nivel"]).reindex(ml.NIVELES_RIESGO)
barras = ax2.bar(niveles.index, niveles.values, color=["#2e7d32", "#ef6c00", "#c62828"])
ax2.bar_label(barras, fmt="%.0f %%")
ax2.axhline(prueba["prevalencia"] * 100, color="black", ls=":", label="Promedio de la prueba")
ax2.set(title="Casos severos según el nivel de riesgo asignado", ylabel="% severos",
        xlabel="Nivel de riesgo")
ax2.legend()
guardar(fig, "fig13_riesgo_prueba")

**Desempeño por subgrupo** (grupos con al menos 50 casos en la prueba):

In [ ]:
subgrupos = pd.DataFrame(prueba["por_subgrupo"])
subgrupos.round(3)

In [ ]:
importancia = permutation_importance(pipeline, X_te, y_te, scoring="roc_auc", n_repeats=10,
                                     random_state=settings.random_state, n_jobs=-1)
importancias = (pd.DataFrame({"variable": X_te.columns,
                              "caida_auc": importancia.importances_mean,
                              "desviacion": importancia.importances_std})
                .sort_values("caida_auc", ascending=False))
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(importancias["variable"][::-1], importancias["caida_auc"][::-1],
        xerr=importancias["desviacion"][::-1], color="#455a64")
ax.set(title="Importancia por permutación (prueba)", xlabel="Caída del AUC al desordenar")
guardar(fig, "fig14_importancia_riesgo")
importancias.round(4)

El AUC de prueba es 0,616 (IC 95 % 0,594-0,637), prácticamente igual al de validación
cruzada (0,62): el modelo no se degrada con los casos más recientes. Es una discriminación
**modesta**: con el umbral de Youden detecta 41 % de los casos severos con especificidad
de 74 %. Sirve para ordenar la lista de seguimiento (en el nivel Alto el 48 % de los casos
fue severo, frente al 27 % en el Bajo), no para decidir sobre un paciente.

Lo que más pesa es la afiliación, los días desde el inicio de los síntomas y la edad; el
sexo, la condición de migrante y el desplazamiento reciente no aportan, y la fiebre
tampoco porque casi todos los casos la tienen.

Por subgrupo, el desempeño es parecido entre sexos, pero el modelo usa la edad y la
afiliación como atajo: en niños de 0 a 4 años casi nunca da la alerta (sensibilidad 2 %),
mientras que a los mayores de 65 años y al régimen subsidiado los marca casi siempre
(especificidad 43 % y 27 %). Esta limitación queda en la ficha del modelo.

## 3. Probabilidad de hospitalización

La clasificación clínica solo viene en los casos del Distrito (2022-2023), pero la
hospitalización (`PAC_HOS`) está en todas las fuentes. Este segundo modelo se entrena con
**todos los casos de 2020 a 2025** y con las variables que traen tanto el INS como el
Distrito: edad, sexo, estrato, afiliación, días desde el inicio de síntomas y condición de
migrante (la comuna, los síntomas y el desplazamiento solo están en 2022-2023).

**Partición temporal:** entrenamiento con los casos notificados en 2020-2023 y prueba con
todos los de 2024-2025, dos años que el modelo nunca vio (incluida la epidemia de 2024).

In [ ]:
Xh, yh, fechas_h = ml.load_dataset_riesgo(objetivo=ml.TARGET_HOSPITALIZACION)
tarea_h = ml.TAREAS_CLASIFICACION[ml.TARGET_HOSPITALIZACION]
Xh_tr, Xh_te, yh_tr, yh_te = ml.split_temporal(Xh, yh, fechas_h,
                                               anio_prueba=tarea_h.anio_prueba)
print("Variables:", tarea_h.variables)
print(f"Entrenamiento: {len(Xh_tr):,} casos 2020-2023 ({yh_tr.mean() * 100:.1f} % "
      f"hospitalizados)")
print(f"Prueba: {len(Xh_te):,} casos 2024-2025 ({yh_te.mean() * 100:.1f} % hospitalizados)")
display(ml.comparar_riesgo(Xh_tr, yh_tr, objetivo=ml.TARGET_HOSPITALIZACION).round(3))

In [ ]:
busqueda_h = ml.buscar_hiperparametros_riesgo(Xh_tr, yh_tr, objetivo=ml.TARGET_HOSPITALIZACION)
print("Mejores parámetros:", busqueda_h.best_params_)
assert busqueda_h.best_params_ == ml.HOSPITALIZACION_PARAMS
pipeline_h, umbrales_h = ml.entrenar_riesgo(Xh_tr, yh_tr, objetivo=ml.TARGET_HOSPITALIZACION)
prueba_h = ml.evaluar_riesgo(pipeline_h, Xh_te, yh_te, umbrales_h)
{k: prueba_h[k] for k in ("roc_auc", "roc_auc_ic95", "sensibilidad", "especificidad",
                          "prevalencia", "n_test", "pct_positivos_por_nivel")}

**¿Qué se gana con los seis años?** Se compara con el mismo modelo entrenado solo con los
casos del Distrito 2022-2023 (como estaba antes), ambos evaluados en 2024-2025:

In [ ]:
solo_distrito = Xh_tr.index[fechas_h[Xh_tr.index].dt.year.between(2022, 2023)]
pipeline_d, _ = ml.entrenar_riesgo(Xh_tr.loc[solo_distrito], yh_tr.loc[solo_distrito],
                                   objetivo=ml.TARGET_HOSPITALIZACION)
grupo_edad_te = grupo_edad(Xh_te["edad"]).astype(str)
calibracion = pd.DataFrame({
    "casos": yh_te.groupby(grupo_edad_te).size(),
    "observado_pct": yh_te.groupby(grupo_edad_te).mean() * 100,
    "todos_los_anios_pct": pd.Series(pipeline_h.predict_proba(Xh_te)[:, 1],
                                     index=Xh_te.index).groupby(grupo_edad_te).mean() * 100,
    "solo_2022_2023_pct": pd.Series(pipeline_d.predict_proba(Xh_te)[:, 1],
                                    index=Xh_te.index).groupby(grupo_edad_te).mean() * 100,
}).reindex(GRUPOS_EDAD)
print(f"AUC en 2024-2025 entrenando con 2020-2023: {prueba_h['roc_auc']:.3f}; "
      f"solo con 2022-2023: {roc_auc_score(yh_te, pipeline_d.predict_proba(Xh_te)[:, 1]):.3f}")
print(f"Casos de entrenamiento: {len(Xh_tr):,} frente a {len(solo_distrito):,}")
calibracion.round(1)

Gradient boosting y Random Forest vuelven a quedar empatados en la validación cruzada
(0,654 y 0,650); se mantiene el Random Forest por coherencia con el primer clasificador.
En la prueba con 38.662 casos de 2024-2025, el AUC es **0,650 (IC 95 % 0,644-0,656)**,
con sensibilidad de 54 % y especificidad de 67 %; en el nivel Alto se hospitalizó el
49 % de los casos, frente al 23 % del Bajo.

Lo que **no** cambia al usar los seis años es la discriminación: el modelo entrenado solo
con 2022-2023 obtiene prácticamente el mismo AUC en 2024-2025 (0,655), y los dos están
igual de bien calibrados por edad (en los mayores de 65 años predicen 46-47 % y se
hospitalizó el 43 %). Más casos no agregan información sobre cada paciente: el techo lo
pone lo poco que se sabe en la primera consulta. Lo que sí se gana es una evaluación
mucho más sólida (dos años completos que el modelo nunca vio, con un intervalo de
confianza tres veces más estrecho que antes, cuando la prueba eran 2.833 casos de 2023)
y un modelo final que aprende de todos los años, no solo de 2022-2023.

Por subgrupo se repite el atajo del primer modelo: en el régimen subsidiado (50 % de
hospitalización) marca a casi todos (especificidad 13 %), y lo mismo con los mayores de
65 años (24 %). Los subsidiados se hospitalizan más por la forma de atención de su red
tanto como por la gravedad, y la hospitalización depende también de las camas y del
criterio de cada IPS: el modelo estima la conducta habitual, no la necesidad clínica.

### Explicación de un caso

Para que el resultado no sea una caja negra, la aplicación muestra qué características del
caso suben o bajan su probabilidad: cada grupo de variables se reemplaza por los valores
de 200 casos reales al azar y se mide cuánto cambia la probabilidad promedio. Por ejemplo,
un paciente de 85 años del régimen subsidiado que consulta al quinto día:

In [ ]:
ejemplo = Xh_te.head(1).assign(edad=85.0, dias_sintomas_consulta=5.0, sexo="F", estrato="2",
                               afiliacion="Subsidiado", migrante=0.0)
fondo_h = ml.muestra_de_fondo(Xh_tr)
print(f"Probabilidad de hospitalización: "
      f"{pipeline_h.predict_proba(ejemplo[ml.RIESGO_VARIABLES])[0, 1] * 100:.0f} % "
      f"(promedio {yh_tr.mean() * 100:.0f} %)")
ml.contribuciones(pipeline_h, ejemplo, fondo_h, tarea_h.variables)[["factor", "puntos"]].round(1)

La probabilidad (72 %) casi duplica el promedio, sobre todo por el régimen subsidiado
(+13 puntos) y la edad (+12,5); el estrato 2 y consultar al quinto día suman algo menos.
Esta es la misma tabla que muestra la aplicación en la pestaña de riesgo.

## 4. Perfiles de comunas (K-means)

Siete indicadores de las 22 comunas urbanas (2022-2023): incidencia, % severos, %
hospitalizados, edad mediana, estrato medio, % régimen subsidiado y hospitalizaciones
infantiles por arbovirosis (centinela 2010-2020). Se estandarizan antes de agrupar.

In [ ]:
comunas = ml.load_dataset_comunas()
seleccion_k = ml.comparar_k(comunas)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(seleccion_k["k"], seleccion_k["silueta"], marker="o", color="#6a1b9a")
ax.set(title="Coeficiente de silueta según el número de grupos", xlabel="k",
       ylabel="Silueta")
guardar(fig, "fig15_silueta_k")
seleccion_k.round(3)

k = 3 tiene la mayor silueta (0,307). Es una estructura moderada: los grupos se
distinguen, pero hay comunas cerca de la frontera entre dos perfiles.

In [ ]:
pipeline_comunas, perfiles = ml.ajustar_perfiles(comunas)
print("Silueta:", round(perfiles.attrs["silueta"], 3))
for grupo, descripcion in pipeline_comunas.descripciones_.items():
    miembros = perfiles.loc[perfiles["grupo"] == grupo, "comuna"].tolist()
    print(f"Grupo {grupo} ({descripcion}): comunas {miembros}")
perfiles.groupby("grupo")[ml.VARIABLES_COMUNAS].mean().round(1)

In [ ]:
Z = pipeline_comunas["scaler"].transform(comunas[ml.VARIABLES_COMUNAS])
pca = PCA(n_components=2, random_state=settings.random_state).fit(Z)
coordenadas = pca.transform(Z)
colores = {"A": "#c62828", "B": "#1565c0", "C": "#6a1b9a"}
fig, ax = plt.subplots(figsize=(7, 5.5))
for grupo, datos in perfiles.groupby("grupo"):
    idx = datos.index
    ax.scatter(coordenadas[idx, 0], coordenadas[idx, 1], s=80, color=colores[grupo],
               label=f"Grupo {grupo}")
    for i in idx:
        ax.annotate(str(perfiles.loc[i, "comuna"]), coordenadas[i], fontsize=8,
                    xytext=(4, 3), textcoords="offset points")
varianza = pca.explained_variance_ratio_ * 100
ax.set(title="Comunas en las dos primeras componentes principales",
       xlabel=f"Componente 1 ({varianza[0]:.0f} %)", ylabel=f"Componente 2 ({varianza[1]:.0f} %)")
ax.legend()
guardar(fig, "fig16_perfiles_comunas")

- **Grupo A — mayor incidencia y severidad** (comunas 3, 4, 6, 7, 8, 9 y 11): 850 casos
  por 100.000, 42 % severos y la mayor carga histórica de hospitalizaciones infantiles.
  Prioridad para control larvario, fumigación y seguimiento de casos.
- **Grupo B — estratos medios-altos, pacientes mayores** (2, 5, 10, 17, 19 y 22): estrato
  medio 3,4, edad mediana 24 años, 9 % subsidiado. Campañas de consulta temprana para
  adultos.
- **Grupo C — estratos bajos, población joven** (1, 12, 13, 14, 15, 16, 18, 20 y 21):
  estrato 2,0, 32 % subsidiado, menor incidencia notificada (428). Queda abierta la pregunta de si
  la incidencia baja es real o refleja menos acceso a la consulta.

Son los mismos tres grupos que se plantearon en la propuesta.

## 5. Decisiones que quedan congeladas en `flows/05_ml_models.py`

| Decisión | Valor | Evidencia |
|---|---|---|
| Pronóstico por horizonte | ingenuo (h=1), tendencia 25 % (h=2-3), tendencia 50 % (h=4) | Menor MAE en la validación 2023, con empate al 1 % a favor del más simple |
| Clima en el pronóstico | Candidato evaluado, no elegido | Empata con la tendencia sin clima |
| Intervalo de pronóstico | 80 %, cuantiles de los errores de validación | Cobertura real en prueba 61-67 % (limitación) |
| Alerta de brote | pronóstico > Q3 del canal endémico | Sensibilidad 96-98 % en 2024-2025 |
| Clasificador de riesgo | Random Forest: 400 árboles, profundidad 10, hojas ≥ 20, 20 % de variables | Validación cruzada y búsqueda en malla |
| Umbral | Youden con predicciones fuera de muestra | Sensibilidad 41 %, especificidad 74 % en prueba |
| Clasificador de hospitalización | El mismo diseño, objetivo `hospitalizado` | AUC 0,647 en prueba |
| Perfiles de comunas | K-means, k = 3, variables estandarizadas | Mayor silueta (0,307) |

`make train` vuelve a ejecutar todo esto, guarda los modelos con sus metadatos en
`models/` y avisa si la validación ya no elige lo mismo que está congelado.